In [12]:
import torch
import numpy as np
import pandas as pd
from model import TimeSeriesTransformer
from sklearn.preprocessing import StandardScaler
import joblib
from fetch import fetch_and_return
from datetime import datetime, timedelta


MODEL_PATH = "models/version2.pth"
SCALER_PATH = "scalers/feature_scaler.pkl"
SEQ_LEN = 24
FEATURES = [
    "ALLSKY_SFC_SW_DIFF", "ALLSKY_SFC_SW_DNI", "TOA_SW_DWN",
    "RH2M", "QV2M", "PS", "WS2M", "CLOUD_AMT",
    "ALLSKY_SFC_LW_DWN", "T2M", "hour_sin", "hour_cos", "month_sin", "month_cos"
]
TARGET = "ALLSKY_SFC_SW_DWN"

def add_time_features(df):
    df['hour'] = df.index.hour
    df['month'] = df.index.month
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    return df

def get_last_24h(lat, lon, location):
    REAL_API_FEATURES = [
        "ALLSKY_SFC_SW_DIFF", "ALLSKY_SFC_SW_DNI", "TOA_SW_DWN",
        "RH2M", "QV2M", "PS", "WS2M", "CLOUD_AMT",
        "ALLSKY_SFC_LW_DWN", "T2M", "ALLSKY_SFC_SW_DWN"
    ]

    all_data = {}
    for param in REAL_API_FEATURES:
        df = fetch_and_return(param, lat, lon, location, days_ago=4)
        df = df[0]
        df = df.replace(-999.0, 0)
        all_data[param] = df.set_index("datetime")[param]

    df_full = pd.concat(all_data.values(), axis=1)
    df_full.columns = REAL_API_FEATURES
    df_full.index = pd.to_datetime(df_full.index)
    df_full = df_full.sort_index()

    # Add derived time features
    df_full = add_time_features(df_full)
    df_full = slice_24h_from_hour(df_full,datetime.now().hour)
    return df_full
def slice_24h_from_hour(df, target_hour=0):
    # Ensure datetime index
    df = df.sort_index()

    # Filter where hour == target_hour (e.g., 0 for 00:00)
    matching_hours = df[df.index.hour == target_hour]

    if len(matching_hours) < 2:
        raise ValueError("Not enough occurrences of target hour to slice 24 hours.")

    # Get first and second occurrence timestamps
    start_time = matching_hours.index[0]
    end_time = matching_hours.index[1]

    # Slice between them
    df_24h = df[(df.index >= start_time) & (df.index < end_time)]

    if len(df_24h) != 24:
        raise ValueError(f"Expected 24 rows, got {len(df_24h)}. Time gap may not be hourly.")

    return df_24h


def prepare_input(df, scaler):
    df = df[FEATURES]
    df_scaled = scaler.transform(df)
    tensor = torch.tensor(df_scaled[-SEQ_LEN:], dtype=torch.float32).unsqueeze(0)
    return tensor

def predict(model, input_tensor):
    with torch.no_grad():
        output = model(input_tensor).item()
        return output




scaler = joblib.load(SCALER_PATH)
model = TimeSeriesTransformer(input_size=len(FEATURES))
model.load_state_dict(torch.load(MODEL_PATH, map_location='cpu'))
model.eval()







C:\Users\grins\AppData\Local\Temp\ipykernel_18184\267779654.py:92: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(MODEL_PATH, map_location='c

TimeSeriesTransformer(
  (input_proj): Linear(in_features=14, out_features=128, bias=True)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=2048, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=2048, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (fc): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): GELU(approximate='none')
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=64, out_features

In [4]:
df = get_last_24h(77.2090, 28.6139, "delhi")

🔍 Fetching ALLSKY_SFC_SW_DIFF from 4 days ago for delhi
2025-07-29 18:45:28.269889
2025-07-31 18:45:28.269889
20250729 20250731
https://power.larc.nasa.gov/api/temporal/hourly/point?parameters=ALLSKY_SFC_SW_DIFF&community=RE&latitude=77.209&longitude=28.6139&start=20250729&end=20250731&format=JSON
📡 Request: https://power.larc.nasa.gov/api/temporal/hourly/point?parameters=ALLSKY_SFC_SW_DIFF&community=RE&latitude=77.209&longitude=28.6139&start=20250729&end=20250731&format=JSON
🔍 Fetching ALLSKY_SFC_SW_DNI from 4 days ago for delhi
2025-07-29 18:45:32.226966
2025-07-31 18:45:32.226966
20250729 20250731
https://power.larc.nasa.gov/api/temporal/hourly/point?parameters=ALLSKY_SFC_SW_DNI&community=RE&latitude=77.209&longitude=28.6139&start=20250729&end=20250731&format=JSON
📡 Request: https://power.larc.nasa.gov/api/temporal/hourly/point?parameters=ALLSKY_SFC_SW_DNI&community=RE&latitude=77.209&longitude=28.6139&start=20250729&end=20250731&format=JSON
🔍 Fetching TOA_SW_DWN from 4 days ago for

In [8]:
df.head()

,ALLSKY_SFC_SW_DIFF,ALLSKY_SFC_SW_DNI,TOA_SW_DWN,RH2M,QV2M,PS,WS2M,CLOUD_AMT,ALLSKY_SFC_LW_DWN,T2M,ALLSKY_SFC_SW_DWN,hour,month,hour_sin,hour_cos,month_sin,month_cos
datetime,,,,,,,,,,,,,,,,,
2025-07-29 18:00:00,0.0,0.0,0.0,98.20,5.31,101.26,1.03,0.0,0.0,5.08,0.0,18,7,-1.000000,-1.836970e-16,-0.5,-0.866025
2025-07-29 19:00:00,0.0,0.0,0.0,98.01,5.32,101.29,1.75,0.0,0.0,5.14,0.0,19,7,-0.965926,2.588190e-01,-0.5,-0.866025
2025-07-29 20:00:00,0.0,0.0,0.0,97.95,5.34,101.35,2.37,0.0,0.0,5.21,0.0,20,7,-0.866025,5.000000e-01,-0.5,-0.866025
2025-07-29 21:00:00,0.0,0.0,0.0,98.07,5.36,101.38,2.56,0.0,0.0,5.25,0.0,21,7,-0.707107,7.071068e-01,-0.5,-0.866025
2025-07-29 22:00:00,0.0,0.0,0.0,98.16,5.38,101.38,2.69,0.0,0.0,5.29,0.0,22,7,-0.500000,8.660254e-01,-0.5,-0.866025


In [6]:
if len(df) < SEQ_LEN:
        print("❌ Not enough data (need at least 24 hourly samples).")

    # Show input time window
start_time = df.index[-SEQ_LEN]
end_time = df.index[-1]
target_time = end_time + pd.Timedelta(hours=1)
print(f"\n🕒 Model input data range: {start_time} → {end_time}")
print(f"🔮 Prediction is for hour: {target_time}")


🕒 Model input data range: 2025-07-29 18:00:00 → 2025-07-30 17:00:00
🔮 Prediction is for hour: 2025-07-30 18:00:00


In [13]:
input_tensor = prepare_input(df, scaler)

a:\weather_net\.venv\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


In [16]:
predict(model,input_tensor)

15.049553871154785